In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


In [2]:
df= pd.read_csv('cars.csv')

In [3]:
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [4]:
pd.get_dummies(df,columns=['fuel','owner'],dtype=int)

,brand,km_driven,selling_price,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,0,1,0,0,1,0,0,0,0
1,Skoda,120000,370000,0,1,0,0,0,0,1,0,0
2,Honda,140000,158000,0,0,0,1,0,0,0,0,1
3,Hyundai,127000,225000,0,1,0,0,1,0,0,0,0
4,Maruti,120000,130000,0,0,0,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,0,0,0,1,1,0,0,0,0
8124,Hyundai,119000,135000,0,1,0,0,0,1,0,0,0
8125,Maruti,120000,382000,0,1,0,0,1,0,0,0,0
8126,Tata,25000,290000,0,1,0,0,1,0,0,0,0


In [5]:
df.shape

(8128, 5)

## K-1 Encoding

In [6]:
pd.get_dummies(df,columns=['fuel','owner'],dtype=int,drop_first=True)

# by doing this the first column from both the categories will be removed 
# to avoid the dummy variable trap and eliminate perfect multicollinearity

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,1,0,0,0,0,0,0
1,Skoda,120000,370000,1,0,0,0,1,0,0
2,Honda,140000,158000,0,0,1,0,0,0,1
3,Hyundai,127000,225000,1,0,0,0,0,0,0
4,Maruti,120000,130000,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,0,0,1,0,0,0,0
8124,Hyundai,119000,135000,1,0,0,1,0,0,0
8125,Maruti,120000,382000,1,0,0,0,0,0,0
8126,Tata,25000,290000,1,0,0,0,0,0,0


but in ML we cannot use pandas for OneHotEncoding because it does not remember the columns sequence 
or category mappings during inference, leading to feature mismatch during test time or model deployment. Instead, use sklearn's OneHotEncoder to fit, save, and transform training and testing data consistently.


In [7]:
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [11]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(df[['brand','km_driven','fuel','owner']],df['selling_price'],test_size=0.3)


In [14]:
from sklearn.preprocessing import OneHotEncoder

In [15]:
ohe=OneHotEncoder()

In [16]:
x_train_new = ohe.fit_transform(x_train[['fuel','owner']]).toarray()

In [18]:
x_test_new = ohe.fit_transform(x_test[['fuel','owner']]).toarray()

# here we have applied OHEon the two columns separately laiter on we will append the remaining two columns in this array

In [26]:
np.hstack((x_train[['brand','km_driven']].values,x_train_new))

array([['Renault', 5000, 0.0, ..., 0.0, 0.0, 0.0],
       ['Tata', 70000, 0.0, ..., 0.0, 0.0, 0.0],
       ['Maruti', 69779, 0.0, ..., 0.0, 0.0, 0.0],
       ...,
       ['Maruti', 90000, 0.0, ..., 1.0, 0.0, 0.0],
       ['Maruti', 58343, 0.0, ..., 0.0, 0.0, 0.0],
       ['Mahindra', 120000, 0.0, ..., 1.0, 0.0, 0.0]],
      shape=(5689, 11), dtype=object)

# droping the first column from both the categories(fuel and owner)

In [23]:
ohe= OneHotEncoder(drop='first',dtype=np.int32)

# sometimes there are too much categories in a column so for we will can push those categories into a separate column which are sued rarely that is here in car dataset some comapnies has alot of cars while some have very less number of cars so in order minimise the complexity we make a new column for those categories

In [29]:
count = df['brand'].value_counts()

In [30]:
count

brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Mitsubishi         14
Force               6
Land                6
Isuzu               5
Kia                 4
Ambassador          4
Daewoo              3
MG                  3
Ashok               1
Opel                1
Peugeot             1
Name: count, dtype: int64

# here we can see that some brands have very less numbers of cars so will make a separate comlum for it

In [31]:
threshold = 100

In [32]:
replace = count[count <= threshold].index

In [33]:
replace

Index(['Nissan', 'Jaguar', 'Volvo', 'Datsun', 'Mercedes-Benz', 'Fiat', 'Audi',
       'Lexus', 'Jeep', 'Mitsubishi', 'Force', 'Land', 'Isuzu', 'Kia',
       'Ambassador', 'Daewoo', 'MG', 'Ashok', 'Opel', 'Peugeot'],
      dtype='object', name='brand')

In [ ]:
pd.get_dummies(df['brand'].replace(replace,'others'), dtype=int)

,BMW,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Renault,Skoda,Tata,Toyota,Volkswagen,others
0,0,0,0,0,0,0,1,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,1,0,0,0,0
2,0,0,0,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8123,0,0,0,0,1,0,0,0,0,0,0,0,0
8124,0,0,0,0,1,0,0,0,0,0,0,0,0
8125,0,0,0,0,0,0,1,0,0,0,0,0,0
8126,0,0,0,0,0,0,0,0,0,1,0,0,0
